# Win Model Evaluation

Overall metrics (single split + CV), then trajectory analysis showing
how the model's ability to identify the winner evolves episode-by-episode.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
from src.models.win import (
    train_eval_pipeline, cross_validate,
    metrics_by_episode_number, winner_rank_detail,
    predict_season,
)
from src.features.build import build_modeling_table
from src.load import load_data


In [ ]:
data = load_data()
df = build_modeling_table(data)

## Overall metrics

Single train/test split (seasons 1-40 → 41-49) and expanding-window CV.
These give one number averaging over all episodes equally.

In [ ]:
print("=== Single split ===\n")
results = train_eval_pipeline(df)

print("\n\n=== Cross-validation ===\n")
cv_results = cross_validate(df)

## Accuracy trajectory (cross-validated)

In [ ]:
# Pool predictions from all CV folds (non-overlapping test seasons → ~49 seasons total)
all_cv_preds = pd.concat([fold["predictions"] for fold in cv_results["folds"]])

# Get accuracy metrics at each episode number, averaged across seasons
by_ep_cv = metrics_by_episode_number(all_cv_preds)
by_ep_cv = by_ep_cv[by_ep_cv["n_seasons"] >= 10]  # drop noisy tail episodes

MODEL_COLOR = "#0d9488"
BASELINE_COLOR = "#9ca3af"
marker_sizes = by_ep_cv["n_seasons"] * 4  # bigger dot = more seasons behind it
z95 = 1.96

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Win model accuracy over season (cross-validated)", fontsize=13, y=1.02)

# Left plot: where does the model rank the winner? (lower = better)
ax = axes[0]
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    by_ep_cv["mean_winner_rank"] - z95 * by_ep_cv["se_winner_rank"],
    by_ep_cv["mean_winner_rank"] + z95 * by_ep_cv["se_winner_rank"],
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_baseline_rank"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Mean winner rank (lower = better)")
ax.set_title("Winner rank")
ax.legend()
ax.invert_yaxis()

# Right plot: how often is the winner the model's #1 pick?
ax = axes[1]
ax.plot(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    (by_ep_cv["top1_accuracy"] - z95 * by_ep_cv["se_top1"]).clip(0),
    (by_ep_cv["top1_accuracy"] + z95 * by_ep_cv["se_top1"]).clip(0, 1),
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["baseline_top1"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Top-1 accuracy")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()

fig.tight_layout()
plt.show()

print(by_ep_cv.to_string(index=False))

## Significance test

In [ ]:
# Reuse the per-(season, episode) detail already computed in win.py
detail = winner_rank_detail(all_cv_preds)

# Is the model's rank improvement over random significantly > 0?
t, p = stats.ttest_1samp(detail["rank_improvement"], 0)
print(f"Paired t-test (model rank vs baseline rank, all season-episodes):")
print(f"  Mean improvement: {detail['rank_improvement'].mean():.2f} ranks better than random")
print(f"  t = {t:.2f}, p = {p:.4f}, n = {len(detail)}")

## Season predictions

Train both models on all prior seasons, then predict a single target season.
Change `TARGET_SEASON` below to view a different season.

In [ ]:
TARGET_SEASON = 50

preds = predict_season(df, TARGET_SEASON)

In [ ]:
def plot_season_predictions(preds, prob_col, title, ylabel, highlight_col=None):
    """Interactive line chart of per-player probabilities over episodes."""
    plot_df = preds.copy()
    plot_df["prob_pct"] = (plot_df[prob_col] * 100).round(1)

    # Mark eliminated players in the hover text
    if highlight_col and highlight_col in plot_df.columns:
        plot_df["status"] = plot_df[highlight_col].map({1: "ELIMINATED", 0: ""})
    else:
        plot_df["status"] = ""

    fig = px.line(
        plot_df.sort_values(["castaway", "episode"]),
        x="episode", y=prob_col, color="castaway",
        markers=True,
        title=title,
        labels={prob_col: ylabel, "episode": "Episode", "castaway": "Player"},
        hover_data={"prob_pct": ":.1f", "status": True, prob_col: False},
    )

    # Add X markers for eliminated players
    if highlight_col and highlight_col in plot_df.columns:
        elim = plot_df[plot_df[highlight_col] == 1]
        if not elim.empty:
            fig.add_scatter(
                x=elim["episode"], y=elim[prob_col],
                mode="markers", marker=dict(symbol="x", size=14, color="black", line_width=2),
                name="Eliminated", showlegend=True,
                hoverinfo="skip",
            )

    fig.update_yaxes(tickformat=".0%")
    fig.update_xaxes(dtick=1)
    fig.update_layout(height=500, legend_title_text="Player")
    fig.show()


plot_season_predictions(
    preds, "prob_eliminated",
    title=f"Season {TARGET_SEASON} — Elimination risk by episode",
    ylabel="P(eliminated)",
    highlight_col="eliminated_this_episode",
)

plot_season_predictions(
    preds, "prob_win",
    title=f"Season {TARGET_SEASON} — Win probability by episode",
    ylabel="P(win)",
    highlight_col="eliminated_this_episode",
)